# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)

---
**Tool disclosure (required):** An LLM was used to help structure this proposal; all design, analysis, and conclusions are our own.


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Divine Zacharias | — |
| Member | Franz Großmann | — |


*Student IDs omitted per the submission guide. Confirm this matches your Moodle group.*


## 2. Mission Title & Research Question

**Title:** *Fatigue, Form, and the Efficiency of Tennis Rankings — Predicting ATP Match Outcomes (1991–2024)*

**Research question:** Using ATP tour-level matches since 1991, we ask: **how accurately can a match's winner be predicted from pre-match information, and can a model beat the "higher-ranked player wins" benchmark?** Beating that benchmark would mean the ranking leaves exploitable information on the table; failing to beat it would mean the ranking is already a near-efficient summary of player quality.

To answer this well, two supporting analyses feed the prediction task:
- a **causal** check of whether *fatigue* (a long previous-round match) genuinely lowers next-match win probability — i.e. whether it is a real driver worth encoding as a feature rather than a spurious correlate; and
- an **unsupervised** decomposition of players into *playing-style archetypes*, used both as predictive features and to characterise *how* different player types win.

**Why it matters:** The ranking acts as the market's consensus signal of player quality, so asking whether a model can out-predict it is an *information-efficiency* test, directly analogous to market-efficiency tests in finance. The fatigue sub-analysis additionally speaks to tournament scheduling and athlete welfare — a mechanism-design question for the sport's multi-billion-euro economy.


## 3. Data

**Source:** Jeff Sackmann's **`tennis_atp`** database (CC BY-NC-SA, attribution required): https://github.com/JeffSackmann/tennis_atp — yearly `atp_matches_YYYY.csv` files. Tour-level match statistics are available from **1991** onward.

**Unit of observation:** one ATP match → reshaped to one **player-match** (player vs. opponent) for modelling.

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| tourney_id | categorical (string) | instrument / metadata | Unique tournament identifier (includes year, level, host city).
| tourney_name | categorical (string) | instrument / metadata | Tournament name (e.g., "Wimbledon").
| surface | categorical (nominal) | feature / control | Court surface (Hard, Clay, Grass, Carpet) affecting play style.
| draw_size | numerical (integer) | feature / control | Tournament draw size (e.g., 32, 64, 128).
| tourney_date | date (YYYYMMDD integer) | instrument / time index | Tournament start date used for chronology and feature lags.
| winner_id | categorical (ID) | instrument / identifier | Winner player ID (used for reshaping to player/opponent).
| winner_ioc | categorical (nominal) | feature / control | Winner nationality (IOC country code).
| winner_ht | numerical (cm) | feature | Winner height in centimeters.
| winner_age | numerical (years) | feature | Winner age at match time (years).
| loser_id | categorical (ID) | instrument / identifier | Loser player ID (used for reshaping to player/opponent).
| loser_name | categorical (string) | instrument / metadata | Loser full name (human-readable label).
| loser_ht | numerical (cm) | feature | Loser height in centimeters.
| loser_ioc | categorical (nominal) | feature / control | Loser nationality (IOC country code).
| loser_age | numerical (years) | feature | Loser age at match time (years).
| score | categorical (string) | instrument / metadata | Match score string (set-by-set results).
| minutes | numerical (integer) | feature | Match duration in minutes (proxy for fatigue/effort).
| winner_rank | numerical (integer) | feature | Winner ATP rank at tournament start (lower is better).
| winner_rank_points | numerical (integer) | feature | Winner ATP ranking points at tournament start.
| loser_rank | numerical (integer) | feature | Loser ATP rank at tournament start.
| loser_rank_points | numerical (integer) | feature | Loser ATP ranking points at tournament start.


**Data quality (critical):**
- Systematic missing values in player entry, seed, and some match stats
- Selection and structural bias from winner/loser-based encoding and from observing only completed matches, which distorts pre-match prediction signals; we address it by reshaping each match into a **symmetric player-vs-opponent** record with **randomised** labels (otherwise a model trivially reads the winner column).
- Historical and temporal inconsistency across 1991–2024, including COVID-era disruptions (2020–21) and incomplete recent seasons.
- Data quality errors and outliers, e.g., impossible player heights and extreme match durations, suggesting measurement or parsing issues.
- Class and category imbalance, especially across surfaces, rounds, and tournament levels, which biases model learning toward dominant conditions (e.g., hard courts, early rounds).

In [ ]:
# Data loading & first inspection (proposal stage)
# Clone https://github.com/JeffSackmann/tennis_atp and place atp_matches_*.csv in ./data/
import pandas as pd, numpy as np, glob

files = sorted(glob.glob("data/atp_matches_*.csv"))
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df = df[df.tourney_date // 10000 >= 1991]            # match stats available from 1991
print("rows, cols:", df.shape)
print("first columns:", df.columns.tolist()[:25])

# Basic data overview / distributions
df["year"] = df["tourney_date"] // 10000
print("year range:", int(df["year"].min()), "-", int(df["year"].max()))

missing = df.isna().mean().sort_values(ascending=False).head(10)
print("\nTop 10 missing-value ratios:")
print(missing)

def show_counts(col, n=10):
    if col in df.columns:
        print(f"\n{col} value counts (top {n}):")
        print(df[col].value_counts(dropna=False).head(n))

show_counts("surface")
show_counts("round")
show_counts("tourney_level")
show_counts("best_of")

numeric_cols = df.select_dtypes(include=[np.number]).columns
key_numeric = [c for c in [
    "minutes",
    "winner_rank", "loser_rank",
    "winner_age", "loser_age",
    "winner_ht", "loser_ht"
] if c in numeric_cols]

print("\nNumeric summary (key columns):")
print(df[key_numeric].describe().T if key_numeric else df[numeric_cols].describe().T)

print("\nYear distribution (matches per year):")
print(df["year"].value_counts().sort_index())

rows, cols: (79172, 49)
first columns: ['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of']
year range: 2000 - 2026

Top 10 missing-value ratios:
winner_entry    0.871381
loser_entry     0.792414
loser_seed      0.768858
winner_seed     0.584070
minutes         0.109988
w_1stWon        0.087278
w_1stIn         0.087278
w_svpt          0.087278
w_df            0.087278
w_ace           0.087278
dtype: float64

surface value counts (top 10):
surface
Hard      43461
Clay      25619
Grass      8000
Carpet     2039
NaN          53
Name: count, dtype: int64

round value counts (top 10):
round
R32     24692
R16     13424
R64     11941
R128     8608
RR       8505
QF       6751
SF       3456
F        1754
ER   

## 4. Planned Methods

Apply at least one technique from **each** block. All three serve the single prediction-and-efficiency question above.

### 4a. Causal Inference
- [x] Causal graph / DAG (DoWhy)  - [x] Backdoor adjustment  - [x] Propensity score stratification

*Why:* before trusting *fatigue* as a predictive feature, we test whether a long previous match **causally** lowers next-match win probability rather than merely correlating with it. A DAG makes our confounding assumptions explicit (both players' rank, surface, round, age, rest days), and backdoor adjustment / propensity-score stratification approximate that effect from observational data.

### 4b. Supervised Learning
- [x] Logistic regression  - [x] Decision Tree / Random Forest

*Why:* this block answers the research question directly. Logistic regression is an interpretable baseline for win prediction; a random forest captures nonlinear interactions and yields feature importances. Both are benchmarked against the "higher-ranked player wins" rule to test ranking efficiency.

### 4c. Unsupervised Learning
- [x] K-Means  - [x] Hierarchical clustering

*Why:* clustering players (per season) on serve/return style surfaces interpretable **archetypes** that double as predictive features and describe *how* player types win. K-Means (silhouette-selected k) is primary; hierarchical clustering is a robustness check and yields an interpretable dendrogram.


## 5. Evaluation Strategy

**Causal (supporting):** DoWhy estimate of `long_prev_match` → next-match win (ATE with confidence interval, in percentage points), with refutation tests (placebo treatment, random-common-cause, data-subset). A robust non-zero effect justifies keeping fatigue as a predictive feature; we acknowledge residual unobserved confounding (e.g. playing style, injury) as a limitation.

**Supervised (primary):** symmetric player-vs-opponent features with randomised labels (no winner/loser leakage); a **chronological** train/test split (train on earlier years, test on later) plus k-fold cross-validation; reported with **accuracy, log-loss, and AUC**. Success is defined relative to the **higher-ranked-player baseline (~65–70%)**: we compare logistic regression vs. random forest and report feature importances. Beating the baseline is direct evidence the ranking is *not* fully information-efficient.

**Unsupervised (supporting):** aggregate per-player-season style features; k chosen by silhouette score (elbow cross-check) and validated via PCA visualisation; each cluster is interpreted (e.g. "big server", "returner", "all-court", "clay specialist") and tracked over eras. Archetype membership is then available as an input to the predictive model.

**Synthesis:** the supervised model answers the headline question — how predictable outcomes are and whether the ranking is an efficient signal — while the causal estimate validates *fatigue* as a genuine driver worth encoding, and the archetypes explain *how* different player types win. Together they turn a single accuracy number into an interpretable account of where the ranking's information gaps come from.


## 6. Work Plan

| # | Owner | Task |
|---|-------|------|
| 1 | Divine | Load 1991+ matches; reshape to symmetric player-vs-opponent records with randomised labels; chronological cleaning and outlier handling |
| 2 | Franz | EDA: win rates by surface/round, ranking-gap baseline (~65–70%), fatigue and form prevalence |
| 3 | Franz | Feature engineering: recent form, surface win-rate, head-to-head, rest days, `long_prev_match` (strict chronological ordering — no future leakage) |
| 4 | Divine | Causal: DAG + propensity-score matching / backdoor adjustment (DoWhy) + refutation — validate fatigue as a feature |
| 5 | Franz | Supervised: logistic regression vs. random forest, chronological CV, accuracy/log-loss/AUC vs. baseline, feature importances |
| 6 | Divine | Unsupervised: per-season style features, K-Means + hierarchical clustering, PCA validation, archetype labelling |
| 7 | All | Synthesis: fold causal + archetype findings into the efficiency interpretation; write-up and presentation |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [2]:
# Causal inference analysis

### 7b. Supervised Learning

In [3]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [4]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
